# SentinelAI - hero embedding, computed properly

The shipped `embedding.json` uses **PCA**, because the machine that built it had no
scikit-learn. PC1 + PC2 hold only **35% of the variance**, so 11,257 benign windows
collapse into one Gaussian smear and the attacks hide inside it. That is why the hero
looks dull: there is no structure on screen for the shader to work with.

This notebook recomputes the **horizontal layout** with UMAP on a GPU and writes a
drop-in replacement with an identical schema. Nothing in the web app changes.

## The one rule

Only `x` and `z` may change. **`y` stays the model log-odds.** Height is the score,
which is the only reason the decision threshold can be drawn as an exact plane. If you
let a dimensionality reducer touch `y`, the plane becomes a decoration and the scene
starts lying. The asserts at the bottom enforce this.

## Setup

1. New Kaggle notebook, **Settings > Accelerator > GPU T4 x2**.
2. **+ Add Input > Upload dataset**, upload `artifacts/scored_test_windows.csv`.
3. The notebook finds the CSV by itself; no path editing needed.
4. Run all. Download `embedding.json` from the output pane.


In [ ]:
import numpy as np, pandas as pd, json, os, glob

# Kaggle renames dataset folders (lowercase, hyphenated, sometimes suffixed), so
# the mount path is not predictable. Find the file instead of hardcoding it.
found = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
print('CSV files visible under /kaggle/input:')
for f in found:
    print('   ', f, '(%.1f MB)' % (os.path.getsize(f) / 1e6))

if not found:
    raise SystemExit(
        'Nothing is mounted. In the right sidebar: + Add Input > Datasets > pick '
        'your upload, wait for it to attach, then re-run this cell.'
    )

# Prefer the scored windows file if several datasets are attached.
CSV = next((f for f in found if 'scored' in f.lower()), found[0])
print()
print('using:', CSV)

# These constants come from the trained pipeline. Do not invent new ones.
TH    = 0.6303419959358633   # operating threshold on probability (budget = 50 alerts/day)
SCALE = 0.42                 # log-odds -> world units, keeps the cloud a sane height
DROP  = {'entity','win','attack','label','baseline_cutoff','probability'}

df = pd.read_csv(CSV)
print('rows', len(df), 'cols', df.shape[1])
assert 'probability' in df.columns and 'label' in df.columns, (
    'That CSV is not the scored test windows file - it has no probability/label column.'
)


## 1. Feature matrix

Identical preprocessing to the shipped generator, so the only variable under test is the
projection itself. Heavy-tailed byte and packet counters get a signed `log1p` or they
dominate every distance computation.


In [ ]:
feat = [c for c in df.columns if c not in DROP]
X = df[feat].astype('float64').to_numpy()
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

heavy = np.abs(X).max(axis=0) > 1000
X[:, heavy] = np.sign(X[:, heavy]) * np.log1p(np.abs(X[:, heavy]))
print('log1p applied to', int(heavy.sum()), 'of', len(feat), 'columns')

mu, sd = X.mean(0), X.std(0)
sd[sd == 0] = 1.0
X = np.clip((X - mu) / sd, -5, 5).astype('float32')
print('feature matrix', X.shape)


## 2. UMAP

cuML's UMAP runs on the T4 and finishes 11k x 40 in seconds. If the GPU import fails the
CPU version gives the same result, just slower.

Knobs worth tuning if you dislike the look:

- `n_neighbors` **low (5-15)** = tight isolated clumps; **high (50-200)** = smoother global shape.
- `min_dist` **low (0.0-0.05)** = dense filaments, dramatic; **high (0.5)** = evenly spread.
- `metric='euclidean'` is fine here since features are already standardised.

For a hero, low `min_dist` looks far better - you want stringy structure, not a lawn.


In [ ]:
import time

N_NEIGHBORS, MIN_DIST, SEED = 25, 0.02, 7

t0 = time.time()
emb2 = None

try:
    from cuml.manifold import UMAP as cuUMAP
    reducer = cuUMAP(n_components=2, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                     random_state=SEED, verbose=True)
    emb2 = np.asarray(reducer.fit_transform(X), dtype='float64')
    print('>>> cuML UMAP on GPU')
except Exception as e:
    print('>>> GPU path unavailable:', type(e).__name__, e)

if emb2 is None:
    import umap
    # Passing random_state to umap-learn forces SINGLE-THREADED execution. That
    # is the usual reason a CPU run takes tens of minutes instead of two. The
    # layout then varies slightly between runs, which is fine: UMAP only sets
    # x and z. Height is the model log-odds, so precision and recall are
    # unaffected and the asserts below still hold exactly.
    reducer = umap.UMAP(n_components=2, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                        n_epochs=200, init='pca', verbose=True)
    emb2 = reducer.fit_transform(X).astype('float64')
    print('>>> umap-learn on CPU (parallel)')

print('embedding', emb2.shape, 'in %.1fs' % (time.time() - t0))


## 3. Fit the world box

The camera path in `hero3d.tsx` is tuned for a cloud roughly 18 units across. Scale by a
robust percentile rather than the max, so a single outlier cannot shrink everything else
into a dot.


In [ ]:
emb2 -= emb2.mean(0)
span = np.percentile(np.abs(emb2), 99)
emb2 = np.clip(emb2 * (9.0 / span), -16, 16)
print('x range', emb2[:,0].min().round(2), emb2[:,0].max().round(2))
print('z range', emb2[:,1].min().round(2), emb2[:,1].max().round(2))


## 4. Height = the model's own log-odds

This is the axis that carries the argument, so it is computed, never learned.


In [ ]:
p = df['probability'].to_numpy('float64')
pc = np.clip(p, 1e-6, 1 - 1e-6)
y  = np.log(pc / (1 - pc)) * SCALE
thresholdY = float(np.log(TH / (1 - TH)) * SCALE)
print('log-odds range %.3f .. %.3f   plane at y = %.4f' % (y.min()/SCALE, y.max()/SCALE, thresholdY))


## 5. Reconcile with the published operating point

If these asserts fail, the projection is not describing the deployed model and must not
be shipped.


In [ ]:
truth = (df['label'].to_numpy() == 1)
above = p >= TH
tp = int((above & truth).sum()); fp = int((above & ~truth).sum())
fn = int((~above & truth).sum()); tn = int((~above & ~truth).sum())
precision = tp / (tp + fp); recall = tp / (tp + fn)
print('above %d = %d true + %d false | missed %d' % (above.sum(), tp, fp, fn))
print('precision %.4f  recall %.4f' % (precision, recall))

assert abs(precision - 0.7959183673469388) < 1e-6, 'precision drifted'
assert abs(recall    - 0.6610169491525424) < 1e-6, 'recall drifted'
print('reconciled with the deployed operating point')


## 6. Write the drop-in file


In [ ]:
fams = ['benign','brute_force','dns_tunnel','dos','exfil','lateral_movement','portscan']
attack = df['attack'].fillna('benign').astype(str).to_numpy()
idx = {f: i for i, f in enumerate(fams)}
fam = [idx.get(a, 0) for a in attack]

pos = np.empty(len(df) * 3, dtype='float64')
pos[0::3] = emb2[:, 0].round(4)
pos[1::3] = np.round(y, 4)
pos[2::3] = emb2[:, 1].round(4)

out = {
    'n': int(len(df)),
    'note': 'x,z = UMAP(n_neighbors=%d, min_dist=%s) of 40 standardised features; y = model log-odds * %s' % (N_NEIGHBORS, MIN_DIST, SCALE),
    'threshold': TH,
    'thresholdY': round(thresholdY, 6),
    'families': fams,
    'counts': {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'above': int(above.sum())},
    'rank1': int(np.argmax(p)),
    'pos': [float(v) for v in pos],
    'p':   [float(v) for v in np.round(p, 6)],
    'fam': fam,
}

with open('embedding.json', 'w') as f:
    json.dump(out, f, separators=(',', ':'))
print('wrote embedding.json  %d bytes  n=%d  rank1=%d' % (os.path.getsize('embedding.json'), out['n'], out['rank1']))


## 7. Install it

Download `embedding.json` from the Kaggle output pane, then on your machine:

```powershell
copy embedding.json C:\projects\sentinelai\artifacts\embedding.json
cd C:\projects\sentinelai\web
npm run sync-data
npm run dev
```

Hard-reload the tab. If the cloud looks too sparse or too clumped, change `N_NEIGHBORS`
and `MIN_DIST` in cell 2 and re-run - nothing else needs to move.
